# AgriStress · Quickstart (offline demo)

**ISRO BAH 2026 · Problem Statement 6** — AI-driven crop-type classification,
phenology-aware moisture-stress detection, and 8-day irrigation advisory.

This notebook runs **end-to-end with NO credentials** (no Earth Engine, no
network). It calls `agristress.pipeline.orchestrator.run_demo()` on **synthetic**
data to produce four artefacts:

1. a **crop-type map**,
2. a **moisture-stress map**,
3. an **irrigation-advisory map**, and
4. an NDVI **time-series chart**.

If the `agristress` package (still under active development by another module
owner) is not importable yet, the notebook **falls back to an embedded synthetic
pipeline** so it always runs. The real GEE multi-satellite pipeline is in
`02_gee_pipeline.ipynb` (requires auth).

## 0 · Setup
Only `numpy` is required. `matplotlib` is used for charts if present, otherwise
a text/ASCII summary is shown — the demo never hard-fails on a missing optional dep.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False
    print('[demo] matplotlib not installed — charts will render as text summaries.')

print('numpy', np.__version__, '| matplotlib:', HAVE_MPL)

## 1 · Run the demo pipeline

Try the real package first. `run_demo()` is expected to return a dict with keys
`crop_map`, `stress_map`, `advisory_map`, `ndvi_series` (2-D arrays + a time
series). If unavailable, we synthesise an equivalent result so the rest of the
notebook is identical either way.

In [ ]:
def _synthetic_run_demo(size=64, n_steps=12, seed=42):
    """Self-contained offline stand-in for agristress.pipeline.orchestrator.run_demo.

    Builds a small synthetic command-area scene with 3 crop types, derives a
    phenology-aware NDVI series, a moisture-stress map and an 8-day advisory map.
    Pure numpy; no I/O. Returns the same dict shape the real orchestrator emits.
    """
    rng = np.random.default_rng(seed)
    # --- crop map: 3 blocky fields (0=wheat, 1=rice, 2=cotton) ---
    crop_map = np.zeros((size, size), dtype=int)
    crop_map[: size // 2, :] = 0
    crop_map[size // 2 :, : size // 2] = 1
    crop_map[size // 2 :, size // 2 :] = 2
    crop_map = np.where(rng.random((size, size)) < 0.05, rng.integers(0, 3, (size, size)), crop_map)

    # --- per-crop double-logistic-ish NDVI trajectories over the season ---
    t = np.linspace(0, 1, n_steps)
    def _logistic(tt, peak, s, a, rsp=12, rau=12):
        green = 1 / (1 + np.exp(-rsp * (tt - s)))
        senesce = 1 / (1 + np.exp(-rau * (tt - a)))
        return 0.15 + (peak - 0.15) * (green - senesce)
    profiles = {
        0: _logistic(t, 0.80, 0.25, 0.70),   # wheat: early
        1: _logistic(t, 0.88, 0.40, 0.85),   # rice: late, high peak
        2: _logistic(t, 0.70, 0.35, 0.80),   # cotton
    }
    # NDVI cube (time, y, x)
    ndvi_cube = np.stack([
        np.choose(crop_map, [profiles[0][k], profiles[1][k], profiles[2][k]])
        + rng.normal(0, 0.02, (size, size))
        for k in range(n_steps)
    ])
    ndvi_cube = np.clip(ndvi_cube, 0, 1)

    # --- moisture stress: a dry patch lowers NDVI vs the crop's expectation ---
    yy, xx = np.mgrid[0:size, 0:size]
    dry = np.exp(-(((xx - size * 0.7) ** 2 + (yy - size * 0.3) ** 2) / (2 * (size * 0.12) ** 2)))
    peak_ndvi = ndvi_cube.max(axis=0)
    expected = np.choose(crop_map, [profiles[0].max(), profiles[1].max(), profiles[2].max()])
    # Vegetation Condition Index-style deficit, amplified by the dry patch.
    stress_map = np.clip((expected - peak_ndvi) * 3 + dry, 0, 1)

    # --- advisory: bucket stress into 0..3 (no_irrigation..severe) ---
    advisory_map = np.digitize(stress_map, [0.15, 0.35, 0.6]).astype(int)

    # --- a representative NDVI series (field means per crop) ---
    ndvi_series = {f'crop_{c}': ndvi_cube[:, crop_map == c].mean(axis=1) for c in range(3)}
    return {
        'crop_map': crop_map,
        'stress_map': stress_map,
        'advisory_map': advisory_map,
        'ndvi_series': ndvi_series,
        'time': t,
        'crop_labels': {0: 'wheat', 1: 'rice', 2: 'cotton'},
        'advisory_labels': {0: 'no_irrigation', 1: 'watch', 2: 'deficit', 3: 'severe_deficit'},
        'source': 'synthetic-fallback',
    }


try:
    from agristress.pipeline.orchestrator import run_demo
    result = run_demo()
    result.setdefault('source', 'agristress.pipeline.orchestrator')
    print('[demo] ran agristress.pipeline.orchestrator.run_demo()')
except Exception as exc:
    print(f'[demo] agristress orchestrator unavailable ({type(exc).__name__}); using embedded synthetic demo.')
    result = _synthetic_run_demo()

print('result keys:', sorted(result.keys()))
print('source     :', result['source'])

## 2 · Crop-type map (Head 1)
Colour-coded crop classes. With representative labels the real classifier targets
Overall Accuracy > 85% (validated by an `errorMatrix` — OA + Kappa — in the GEE notebook).

In [ ]:
crop_map = np.asarray(result['crop_map'])
labels = result.get('crop_labels', {0: 'class0', 1: 'class1', 2: 'class2'})

if HAVE_MPL:
    plt.figure(figsize=(4.5, 4))
    plt.imshow(crop_map, cmap='tab10', vmin=0, vmax=9)
    plt.title('Crop-type map')
    plt.axis('off')
    cbar = plt.colorbar(ticks=sorted(labels), shrink=0.8)
    cbar.ax.set_yticklabels([labels[k] for k in sorted(labels)])
    plt.show()
else:
    vals, counts = np.unique(crop_map, return_counts=True)
    print('Crop-type pixel counts:')
    for v, c in zip(vals, counts):
        print(f'  {labels.get(int(v), v):>14}: {c}')

## 3 · Moisture-stress map (Head 2)
Stage-aware stress (0 = healthy → 1 = severe). High values flag a dry patch where
NDVI falls below the crop's phenological expectation.

In [ ]:
stress_map = np.asarray(result['stress_map'])

if HAVE_MPL:
    plt.figure(figsize=(4.5, 4))
    im = plt.imshow(stress_map, cmap='RdYlGn_r', vmin=0, vmax=1)
    plt.title('Moisture-stress index')
    plt.axis('off')
    plt.colorbar(im, shrink=0.8, label='stress (0–1)')
    plt.show()
else:
    print('Moisture stress — min/mean/max: '
          f'{stress_map.min():.2f} / {stress_map.mean():.2f} / {stress_map.max():.2f}')
    print(f'Stressed pixels (>0.5): {(stress_map > 0.5).mean() * 100:.1f}%')

## 4 · Irrigation-advisory map (Head 3)
8-day crop-water-deficit translated into actionable classes for canal command
areas: `no_irrigation → watch → deficit → severe_deficit`.

In [ ]:
advisory_map = np.asarray(result['advisory_map'])
adv_labels = result.get('advisory_labels', {0: 'no_irrigation', 1: 'watch', 2: 'deficit', 3: 'severe_deficit'})

if HAVE_MPL:
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(['#2c7bb6', '#abd9e9', '#fdae61', '#d7191c'])
    plt.figure(figsize=(4.5, 4))
    im = plt.imshow(advisory_map, cmap=cmap, vmin=0, vmax=3)
    plt.title('Irrigation advisory (8-day)')
    plt.axis('off')
    cbar = plt.colorbar(im, ticks=[0, 1, 2, 3], shrink=0.8)
    cbar.ax.set_yticklabels([adv_labels[k] for k in [0, 1, 2, 3]])
    plt.show()
else:
    vals, counts = np.unique(advisory_map, return_counts=True)
    print('Advisory class areas:')
    for v, c in zip(vals, counts):
        print(f'  {adv_labels.get(int(v), v):>16}: {c} px ({c / advisory_map.size * 100:.1f}%)')

## 5 · NDVI time-series (phenology)
Per-crop seasonal NDVI trajectories — the phenology that makes stress detection
and the Kc-from-NDVI water demand growth-stage aware.

In [ ]:
series = result['ndvi_series']
t = np.asarray(result.get('time', np.linspace(0, 1, len(next(iter(series.values()))))))

if HAVE_MPL:
    plt.figure(figsize=(6, 3.5))
    for name, y in series.items():
        plt.plot(t, y, marker='o', ms=3, label=name)
    plt.xlabel('season (0–1)')
    plt.ylabel('NDVI')
    plt.title('Phenology — per-crop NDVI series')
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Per-crop NDVI (sparkline-ish):')
    ramp = ' .:-=+*#%@'
    for name, y in series.items():
        y = np.asarray(y)
        spark = ''.join(ramp[int(np.clip(v, 0, 1) * (len(ramp) - 1))] for v in y)
        print(f'  {name:>8}: |{spark}|  peak={y.max():.2f}')

## 6 · Summary

This offline demo produced all four PS6 deliverables — **crop map**,
**stage-aware moisture stress**, **8-day irrigation advisory**, and a
**phenology time-series** — with zero credentials, exercising the same
result contract the production `agristress.pipeline.orchestrator.run_demo()`
returns.

Next: open **`02_gee_pipeline.ipynb`** to run the real multi-satellite fusion
(Sentinel-1/2, Landsat, SMAP, AlphaEarth embeddings, ERA5-Land) on Google Earth
Engine and export the gold COG/asset artefacts that feed the O(1) tile server.